# Project Bigoni, Meloni, Rossini

# Libraries

In [3]:
# =========================================================
# Core / utilities
# =========================================================
import os
import re
import time
import glob
import sqlite3
import lxml
from datetime import datetime
from io import BytesIO
from zipfile import ZipFile
import xml.etree.ElementTree as ET

# =========================================================
# Data handling
# =========================================================
import pandas as pd
import numpy as np

# =========================================================
# HTTP / API
# =========================================================
import requests
import requests_cache
import openmeteo_requests
from retry_requests import retry

# =========================================================
# HTML parsing (BRef tables in comments)
# =========================================================
from bs4 import BeautifulSoup, Comment

# =========================================================
# Selenium (dynamic scraping)
# =========================================================
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO


# Set Work directory

In [4]:
#set working directory
os.chdir(r"C:\Users\bebab\Desktop\Università\DataManagement\DataManLab\work\DMproject\BikeHistoricalData")
print(os.getcwd())

C:\Users\bebab\Desktop\Università\DataManagement\DataManLab\work\DMproject\BikeHistoricalData


# 1. Data Acquisition
In this section, we connect to different sources to retrieve the raw data.

## 1.1 Bike Data Acquisition

In [5]:
#look for all citibike data files in the current directory
file_list = glob.glob("*-citibike-tripdata_*.csv")
#exclude files that could be not related to bike data (e.g., Analysis files, knicks files)
file_list = [f for f in file_list if "Analysis" not in f and "knicks" not in f]
print(f"Found {len(file_list)} bike data files.")

Found 53 bike data files.


## 1.2 Weather Data Acquisition
We connect to Open-Meteo API to fetch the raw weather records.

In [6]:
# Set up the API connection with caching
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)
# API URL and parameters (asking for a wide range to cover all potential bike data)
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 40.7128, # New York
    "longitude": -74.0060,
    "start_date": "2024-10-31",
    "end_date": "2025-10-31",
    "hourly": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation", "weather_code", "wind_speed_10m"],
    "timezone": "America/New_York"
}
# Request the data
weather_responses = openmeteo.weather_api(url, params=params)
raw_weather_response = weather_responses[0]

## 1.3 Knicks Data Acquisition
We use Selenium to scrape the game schedule from ESPN.

In [7]:

def make_driver(headless=False):  #nuova funzione make_driver perchè non ho chrome nel docker
    options = webdriver.ChromeOptions()
    # Le opzioni che avevi già vanno bene
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    # SE IL CODICE È DENTRO DOCKER, DEVI USARE QUESTO:
    # Ci colleghiamo al container "selenium" che vedo nella tua foto sulla porta 4444
    driver = webdriver.Remote(
        command_executor='http://selenium:4444/wd/hub',
        options=options
    )
    
    return driver

In [8]:

BASE = "https://www.basketball-reference.com"
TEAM = "NYK"
YEARS = (2025, 2026)


#  Driver setup
'''
def make_driver(headless=False):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=options)
'''
#  Cookie banner (multilingual))                 
def accept_cookies_if_present(driver, timeout=6):
    xpaths = [
        "//button[contains(., 'Accetta tutto')]",
        "//button[contains(., 'Accept All')]",
        "//button[contains(., 'Accept all')]",
        "//button[contains(., 'Salva')]",
        "//button[contains(., 'Save')]",
        "//button[contains(., 'Rifiutare')]",
        "//button[contains(., 'Reject')]",
        "//a[contains(., 'Accetta tutto')]",
        "//a[contains(., 'Accept All')]",
                ]
    for xp in xpaths:
        try:
            btn = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            btn.click()
            time.sleep(0.4)
            return True
        except Exception:
            pass
    return False


#  Extract RAW games table (no cleaning)

def extract_games_df_raw(page_html: str) -> pd.DataFrame:
    tables = pd.read_html(StringIO(page_html), attrs={"id": "games"})
    if not tables:
        raise ValueError("table id='games' not found.")
    return tables[0].copy()

# Main scraper (dynamic via Selenium, no click)

def scrape_knicks_games_raw(years=(2025, 2026), headless=False) -> pd.DataFrame:
    driver = make_driver(headless=headless)
    wait = WebDriverWait(driver, 20)

    frames = []
    try:
        for year in years:
            url = f"{BASE}/teams/{TEAM}/{year}_games.html"
            driver.get(url)

            # dynamic sync: page loaded
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            accept_cookies_if_present(driver)

            time.sleep(0.8)

            df_raw = extract_games_df_raw(driver.page_source)

            # minimal metadata only (not domain cleaning)
            df_raw.insert(0, "season_year", year)
            frames.append(df_raw)

        return pd.concat(frames, ignore_index=True)

    finally:
        driver.quit()


# RUN + save (RAW only)

if __name__ == "__main__":
    df_raw_all = scrape_knicks_games_raw(YEARS, headless=False)
    df_raw_all.to_csv("knicks_bref_RAW_dynamic.csv", index=False)

MaxRetryError: HTTPConnectionPool(host='selenium', port=4444): Max retries exceeded with url: /wd/hub/session (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001E966231D10>: Failed to resolve 'selenium' (Name or service not known: selenium using 1 resolver(s))"))

# 2. Pre-processing and Cleaning
Now we take the raw data and clean it up for analysis

## 2.1 Bike Data Cleaning
We iterate through files, filter bad trips, and aggregate by hour.

In [ ]:
all_hourly_data = []

# Loop su ogni file
for file in file_list:
        df = pd.read_csv(file, usecols=['started_at', 'ended_at', 'member_casual'])
        
        # convert all time columns to datetime
        df['started_at'] = pd.to_datetime(df['started_at'])
        df['ended_at'] = pd.to_datetime(df['ended_at'])
        
        #   claculate all trip duration in minutes
        df['duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60
        
        # only trips between 1 minute and 3 hours 
        df = df[(df['duration_min'] > 1) & (df['duration_min'] < 180)].copy() # perche cazzo lo facciamo
        
        # count members and casual
        df['member_count'] = np.where(df['member_casual'].str.contains('member|Subscriber', case=False, na=False), 1, 0)
        df['casual_count'] = np.where(df['member_casual'].str.contains('casual|Customer', case=False, na=False), 1, 0)
        
        # creation of an date and hour column
        df['date']=df['started_at'].dt.date
        df['time_hour']=df['started_at'].dt.floor('h')
        df['time_hour']=df['time_hour'].dt.strftime('%H:%M')

        # Aggregartio per hour (summs)
        hourly_stats = df.groupby(['date', 'time_hour']).agg(
            total_trips=('started_at', 'count'),
            sum_duration_min=('duration_min', 'sum'),
            member_count=('member_count', 'sum'),
            casual_count=('casual_count', 'sum')
        ).reset_index()
        
    
        all_hourly_data.append(hourly_stats)
    
    #except Exception as e:
    #    print("Could not read file:", file)
    #    print("Reason:", e)

df_bike = (pd.concat(all_hourly_data).groupby(['date', 'time_hour'], as_index=False).sum())

# calculate final average duration
df_bike['avg_duration_min'] = df_bike['sum_duration_min'] / df_bike['total_trips']

df_bike = df_bike.sort_values(['date', 'time_hour'])
df_bike['avg_duration_min'] = df_bike['avg_duration_min'].round(2)
df_bike['sum_duration_min'] = df_bike['sum_duration_min'].round(2)

print("Bike data ready! Rows:", len(df_bike))
print("Max average duration:", df_bike['avg_duration_min'].max())

Bike data ready! Rows: 8762
Max average duration: 156.89


In [ ]:
df_bike.to_csv('checkpoitn_bici_8_01.csv')

In [ ]:
df_bike

,date,time_hour,total_trips,sum_duration_min,member_count,casual_count,avg_duration_min
0,2024-10-31,21:00,2,313.79,1,1,156.89
1,2024-10-31,22:00,20,2133.32,5,15,106.67
2,2024-10-31,23:00,981,22281.93,677,304,22.71
3,2024-11-01,00:00,4023,53396.29,2888,1135,13.27
4,2024-11-01,01:00,2667,35169.29,1845,822,13.19
...,...,...,...,...,...,...,...
8757,2025-10-31,19:00,8910,105210.18,7239,1671,11.81
8758,2025-10-31,20:00,7099,85519.13,5601,1498,12.05
8759,2025-10-31,21:00,5735,70232.43,4401,1334,12.25
8760,2025-10-31,22:00,5002,61096.57,3752,1250,12.21


In [ ]:
df_bike=pd.read_csv('checkpoitn_bici_8_01.csv', usecols=['date',	'time_hour','total_trips','sum_duration_min','member_count','casual_count','avg_duration_min'])

In [ ]:
df_bike 

,date,time_hour,total_trips,sum_duration_min,member_count,casual_count,avg_duration_min
0,2024-10-31,21:00,2,313.79,1,1,156.89
1,2024-10-31,22:00,20,2133.32,5,15,106.67
2,2024-10-31,23:00,981,22281.93,677,304,22.71
3,2024-11-01,00:00,4023,53396.29,2888,1135,13.27
4,2024-11-01,01:00,2667,35169.29,1845,822,13.19
...,...,...,...,...,...,...,...
8757,2025-10-31,19:00,8910,105210.18,7239,1671,11.81
8758,2025-10-31,20:00,7099,85519.13,5601,1498,12.05
8759,2025-10-31,21:00,5735,70232.43,4401,1334,12.25
8760,2025-10-31,22:00,5002,61096.57,3752,1250,12.21


## 2.2 Weather Data Cleaning
We process the API response into a clean DataFrame and handle timezones.

In [ ]:
hourly = raw_weather_response.Hourly()
hourly_data = {"date": pd.date_range(
    start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
    end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
    freq = pd.Timedelta(seconds = hourly.Interval()),
    inclusive = "left"
)}

hourly_data["temp_c"] = np.round(hourly.Variables(0).ValuesAsNumpy(), 1)  #temperature
hourly_data["humidity"] = np.round(hourly.Variables(1).ValuesAsNumpy(), 1)
hourly_data["temp_apparent_c"] = np.round(hourly.Variables(2).ValuesAsNumpy(), 1)  #temperature_apparent
hourly_data["rain_mm"] = np.round(hourly.Variables(3).ValuesAsNumpy(), 1) #rain
hourly_data["weather_code"] = hourly.Variables(4).ValuesAsNumpy()   #type_weather
hourly_data["wind_kmh"] = np.round(hourly.Variables(5).ValuesAsNumpy(), 1) #wind

df_weather = pd.DataFrame(data = hourly_data)
df_weather['date'] = df_weather['date'].dt.tz_localize(None) # Remove time zone info to match bike data

In [ ]:

# 1. Pulisci i nomi delle colonne (per sicurezza)
#df_weather.columns = df_weather.columns.str.strip()

# convertion and rounding (fix fort the infinite decimals)
cols_to_fix = ['temp_c', 'humidity', 'temp_apparent_c', 'rain_mm', 'wind_kmh']
df_weather[cols_to_fix] = df_weather[cols_to_fix].astype(float).round(1)

# convert the date and create the primary key (pk_dt)
df_weather['date'] = pd.to_datetime(df_weather['date'])

print(len(df_weather))
df_weather.head()

8784


,date,temp_c,humidity,temp_apparent_c,rain_mm,weather_code,wind_kmh
0,2024-10-31 05:00:00,15.5,81.2,14.8,0.0,2.0,9.7
1,2024-10-31 06:00:00,14.2,85.8,13.6,0.0,2.0,8.7
2,2024-10-31 07:00:00,13.9,86.3,13.1,0.0,2.0,9.5
3,2024-10-31 08:00:00,13.2,88.6,12.5,0.0,3.0,7.8
4,2024-10-31 09:00:00,12.2,92.1,11.3,0.0,2.0,8.6


## 2.3 Knicks Data Cleaning
We ensure dates are correct and categorize games as Home or Away.

In [ ]:

def add_date_time_home_away_from_bref(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # repeted headers inside BRef table -> keeps only matchin Rk numeric rows
    if "Rk" in out.columns:
        out = out[out["Rk"].astype(str).str.match(r"^\d+$", na=False)].copy()

    # 1) HOME / AWAY -> inside the dataset the 'away' maker is in "Unnamed: 5" with value "@"

    away_marker = out["Unnamed: 5"].astype(str).str.strip()
    out["home_away"] = np.where(away_marker.eq("@"), "away", "home")

    #  date -> AAAA/MM/GG

    out["date_dt"] = pd.to_datetime(out["Date"], errors="coerce")
    out["date_yyyy_mm_dd"] = out["date_dt"].dt.strftime("%Y/%m/%d")

    
    # 3) hours -> 24h (HH:MM)
    # Start (ET) esempio: "7:30p", "7:00p"
    
    start_raw = out["Start (ET)"].astype(str).str.strip()

    # converting "7:30p" -> "7:30 PM", "11:00a" -> "11:00 AM"
    start_norm = (
        start_raw
        .str.replace(r"(?i)\s*tbd\s*", "", regex=True)
        .str.replace(r"(?i)a$", " AM", regex=True)
        .str.replace(r"(?i)p$", " PM", regex=True)
        .str.strip()
    )

    t = pd.to_datetime(start_norm, format="%I:%M %p", errors="coerce")
    out["time_24h"] = t.dt.strftime("%H:%M")

    return out


df2 = add_date_time_home_away_from_bref(df_raw_all)

In [ ]:

df_knicks = df2.loc[:, [
    "date_yyyy_mm_dd", "time_24h", "home_away", "Opponent",
    "Tm", "Opp", "W", "L", "Streak", "LOG"
]].copy()

# 1) Rimuovi righe header ripetute (pulizia dati sporchi dallo scraping) 
mask_header = (
    df_knicks["Opponent"].astype(str).str.strip().eq("Opponent") |
    df_knicks["W"].astype(str).str.strip().eq("W")
)
df_knicks = df_knicks.loc[~mask_header].copy()

# 2) Parse datetime base 
dt = pd.to_datetime(
    df_knicks["date_yyyy_mm_dd"].astype(str) + " " + df_knicks["time_24h"].astype(str),
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)

# dropping eventual rows that could not be parsed
df_knicks = df_knicks.loc[dt.notna()].copy()
dt = dt.loc[dt.notna()]

# rounding to the nearest hour (:00–:30 previous, :31–:59 next) -> to allineate with wether and bike data
dt_rounded = dt.dt.floor("h").where(dt.dt.minute <= 30, dt.dt.ceil("h"))

df_knicks["time_24h"] = dt_rounded.dt.strftime("%H:00")

print(df_knicks.head())

  date_yyyy_mm_dd time_24h home_away             Opponent   Tm  Opp  W  L  \
0      2024/10/22    19:00      away       Boston Celtics  109  132  0  1   
1      2024/10/25    19:00      home       Indiana Pacers  123   98  1  1   
2      2024/10/28    19:00      home  Cleveland Cavaliers  104  110  1  2   
3      2024/10/30    19:00      away           Miami Heat  116  107  2  2   
4      2024/11/01    19:00      away      Detroit Pistons  128   98  3  2   

  Streak   LOG  
0    L 1  2:04  
1    W 1  2:21  
2    L 1  2:23  
3    W 1  2:10  
4    W 2  1:55  


In [ ]:
df_knicks

,date_yyyy_mm_dd,time_24h,home_away,Opponent,Tm,Opp,W,L,Streak,LOG
0,2024/10/22,19:00,away,Boston Celtics,109,132,0,1,L 1,2:04
1,2024/10/25,19:00,home,Indiana Pacers,123,98,1,1,W 1,2:21
2,2024/10/28,19:00,home,Cleveland Cavaliers,104,110,1,2,L 1,2:23
3,2024/10/30,19:00,away,Miami Heat,116,107,2,2,W 1,2:10
4,2024/11/01,19:00,away,Detroit Pistons,128,98,3,2,W 2,1:55
...,...,...,...,...,...,...,...,...,...,...
166,2026/04/03,19:00,home,Chicago Bulls,NaN,NaN,NaN,NaN,NaN,NaN
167,2026/04/06,19:00,away,Atlanta Hawks,NaN,NaN,NaN,NaN,NaN,NaN
168,2026/04/09,19:00,home,Boston Celtics,NaN,NaN,NaN,NaN,NaN,NaN
170,2026/04/10,19:00,home,Toronto Raptors,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Data Integration

## 3.1 Primary Key Generation & Alignment

### 3.1.1 Bike data


In [ ]:
# union of date and time_hour into a single datetime column
dt_temp = pd.to_datetime(df_bike['date'].astype(str) + ' ' + df_bike['time_hour'].astype(str))

# rounding, if minutes <=30 round down, else round up
dt_rounded = dt_temp.dt.floor("h").where(dt_temp.dt.minute <= 30, dt_temp.dt.ceil("h"))

# genering. a primary key (pk_dt) in YYYYMMDDHH format
df_bike['pk_dt'] = dt_rounded.dt.strftime("%Y%m%d%H").astype("int64")

# defining the time range to filter basing on the key (YYYYMMDDHH): start Nov 1 at 00;00, 2024 to Oct 31 at 23:00, 2025
pk_start = 2024110100
pk_end = 2025103123

df_bike = df_bike.loc[(df_bike['pk_dt'] >= pk_start) & (df_bike['pk_dt'] <= pk_end)].reset_index(drop=True)

# Controllo veloce
print(df_bike['pk_dt'].min())
print(df_bike['pk_dt'].max())


2024110100
2025103123


In [ ]:
df_bike

,date,time_hour,total_trips,sum_duration_min,member_count,casual_count,avg_duration_min,pk_dt
0,2024-11-01,00:00,4023,53396.29,2888,1135,13.27,2024110100
1,2024-11-01,01:00,2667,35169.29,1845,822,13.19,2024110101
2,2024-11-01,02:00,1690,22771.54,1141,549,13.47,2024110102
3,2024-11-01,03:00,1073,14562.98,755,318,13.57,2024110103
4,2024-11-01,04:00,857,10331.25,616,241,12.06,2024110104
...,...,...,...,...,...,...,...,...
8754,2025-10-31,19:00,8910,105210.18,7239,1671,11.81,2025103119
8755,2025-10-31,20:00,7099,85519.13,5601,1498,12.05,2025103120
8756,2025-10-31,21:00,5735,70232.43,4401,1334,12.25,2025103121
8757,2025-10-31,22:00,5002,61096.57,3752,1250,12.21,2025103122


### 3.1.2 Weather data

In [ ]:
df_weather['pk_dt'] = df_weather['date'].dt.floor('h').dt.strftime("%Y%m%d%H").astype("int64")

#filetr for the analysis period (01/11/2024 00:00 - 31/10/2025 23:00)
pk_start = 2024110100
pk_end = 2025103123

df_weather = df_weather.loc[(df_weather['pk_dt'] >= pk_start) & (df_weather['pk_dt'] <= pk_end)].reset_index(drop=True)


In [ ]:
df_weather

,date,temp_c,humidity,temp_apparent_c,rain_mm,weather_code,wind_kmh,pk_dt
0,2024-11-01 00:00:00,22.8,53.5,21.0,0.0,3.0,18.6,2024110100
1,2024-11-01 01:00:00,22.2,56.4,20.2,0.0,3.0,20.8,2024110101
2,2024-11-01 02:00:00,21.8,59.3,19.8,0.0,3.0,21.2,2024110102
3,2024-11-01 03:00:00,21.6,61.4,19.6,0.0,3.0,22.2,2024110103
4,2024-11-01 04:00:00,21.3,63.1,19.0,0.0,3.0,24.6,2024110104
...,...,...,...,...,...,...,...,...
8755,2025-10-31 19:00:00,13.4,47.0,8.0,0.0,3.0,25.7,2025103119
8756,2025-10-31 20:00:00,13.0,51.9,7.5,0.0,1.0,27.4,2025103120
8757,2025-10-31 21:00:00,12.2,52.3,6.8,0.0,1.0,26.3,2025103121
8758,2025-10-31 22:00:00,11.6,54.0,6.9,0.0,1.0,20.6,2025103122


### 3.1.3 Knicks games data

In [ ]:
# primary key (pk_dt) in YYYYMMDDHH format 
df_knicks["pk_dt"] = dt_rounded.dt.strftime("%Y%m%d%H").astype("int64")

# filter period of interest 
start = pd.Timestamp("2024-11-01")
end   = pd.Timestamp("2025-10-31 23:59:59")

mask_period = (dt_rounded >= start) & (dt_rounded <= end)
df_knicks = df_knicks.loc[mask_period].reset_index(drop=True)


In [ ]:
df_knicks

,date_yyyy_mm_dd,time_24h,home_away,Opponent,Tm,Opp,W,L,Streak,LOG,pk_dt
0,2024/10/30,19:00,away,Miami Heat,116,107,2,2,W 1,2:10,2024110100
1,2024/11/01,19:00,away,Detroit Pistons,128,98,3,2,W 2,1:55,2024110101
2,2024/11/04,21:00,away,Houston Rockets,97,109,3,3,L 1,2:08,2024110102
3,2024/11/06,19:00,away,Atlanta Hawks,116,121,3,4,L 2,2:16,2024110103
4,2024/11/08,19:00,home,Milwaukee Bucks,116,94,4,4,W 1,2:07,2024110104
...,...,...,...,...,...,...,...,...,...,...,...
156,2026/04/03,19:00,home,Chicago Bulls,NaN,NaN,NaN,NaN,NaN,NaN,2024110719
157,2026/04/06,19:00,away,Atlanta Hawks,NaN,NaN,NaN,NaN,NaN,NaN,2024110720
158,2026/04/09,19:00,home,Boston Celtics,NaN,NaN,NaN,NaN,NaN,NaN,2024110721
159,2026/04/10,19:00,home,Toronto Raptors,NaN,NaN,NaN,NaN,NaN,NaN,2024110723


## 3.2 Investigation of Correspondences (Consistency Check)
We check if the datasets cover the same time periods to ensure a good merge.

In [ ]:
print(f"Bike Key Range:    {df_bike['pk_dt'].min()} to {df_bike['pk_dt'].max()}")
print(f"Weather Key Range: {df_weather['pk_dt'].min()} to {df_weather['pk_dt'].max()}")
print(f"Knicks Key Range:  {df_knicks['pk_dt'].min()} to {df_knicks['pk_dt'].max()}")
# Check Weather Coverage
missing_keys = df_bike[~df_bike['pk_dt'].isin(df_weather['pk_dt'])]
if not missing_keys.empty:
    print( len(missing_keys))
else:
    print("total coverage")

Bike Key Range:    2024110100 to 2025103123
Weather Key Range: 2024110100 to 2025103123
Knicks Key Range:  2024110100 to 2024110800
total coverage


## 3.3 Data Consolidation Strategy

### 3.3.1 Data Enrichment: Temporal Window Expansion (Knicks)

In [ ]:
# advanced feature engineering
# expansion temporal window for knicks events
print("--- Espansione Finestra Temporale Knicks ---")
knicks_expanded_list = []

for _, row in df_knicks.iterrows(): 
    game_pk = row['pk_dt']
    base_time = pd.to_datetime(str(game_pk), format='%Y%m%d%H')

    # Window: 1 hour before, during, 1 hour after, 2 hours after
    offsets = [-1, 0, 1, 2]
    for h in offsets:
        new_time = base_time + pd.Timedelta(hours=h)
        new_pk = int(new_time.strftime('%Y%m%d%H'))
        knicks_expanded_list.append({
            'pk_dt': new_pk,
            'is_knicks_event': 1, 
            'opponent': row['Opponent']
        })

df_knicks_EXTENDED= pd.DataFrame(knicks_expanded_list)
# Rimuoviamo duplicati (se due finestre si sovrappongono) -> per un aumento dati futuro viene inserito nel cosice ma effettivamente non
# è  necessario dal momento che il caso preso in analisi è molto specifico (knicks)
df_knicks_EXTENDED = df_knicks_EXTENDED.drop_duplicates(subset=['pk_dt'])
print(f"Knicks espansi: {len(df_knicks_EXTENDED)} ore totali di 'evento'.")

--- Espansione Finestra Temporale Knicks ---
Knicks espansi: 172 ore totali di 'evento'.


In [ ]:
df_knicks_EXTENDED

,pk_dt,is_knicks_event,opponent
0,2024103123,1,Miami Heat
1,2024110100,1,Miami Heat
2,2024110101,1,Miami Heat
3,2024110102,1,Miami Heat
7,2024110103,1,Detroit Pistons
...,...,...,...
631,2024110722,1,Atlanta Hawks
635,2024110723,1,Boston Celtics
638,2024110800,1,Toronto Raptors
639,2024110801,1,Toronto Raptors


### 3.3.2 Execution of Merges (Bike + Weather + Events)

In [ ]:
#final data integration
# bike + mwether 
df_integrated = pd.merge(df_bike, df_weather[['pk_dt', 'temp_c', 'rain_mm', 'wind_kmh']], 
                         on='pk_dt', how='left')

# merge Knicks (temporal window)
df_integrated = pd.merge(df_integrated, df_knicks_EXTENDED, on='pk_dt', how='left')

# manage of null cases
df_integrated['is_knicks_event'] = df_integrated['is_knicks_event'].fillna(0).astype(int)
df_integrated['opponent'] = df_integrated['opponent'].fillna("No Game")


In [ ]:
df_integrated

,date,time_hour,total_trips,sum_duration_min,member_count,casual_count,avg_duration_min,pk_dt,temp_c,rain_mm,wind_kmh,is_knicks_event,opponent
0,2024-11-01,00:00,4023,53396.29,2888,1135,13.27,2024110100,22.8,0.0,18.6,1,Miami Heat
1,2024-11-01,01:00,2667,35169.29,1845,822,13.19,2024110101,22.2,0.0,20.8,1,Miami Heat
2,2024-11-01,02:00,1690,22771.54,1141,549,13.47,2024110102,21.8,0.0,21.2,1,Miami Heat
3,2024-11-01,03:00,1073,14562.98,755,318,13.57,2024110103,21.6,0.0,22.2,1,Detroit Pistons
4,2024-11-01,04:00,857,10331.25,616,241,12.06,2024110104,21.3,0.0,24.6,1,Houston Rockets
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8754,2025-10-31,19:00,8910,105210.18,7239,1671,11.81,2025103119,13.4,0.0,25.7,0,No Game
8755,2025-10-31,20:00,7099,85519.13,5601,1498,12.05,2025103120,13.0,0.0,27.4,0,No Game
8756,2025-10-31,21:00,5735,70232.43,4401,1334,12.25,2025103121,12.2,0.0,26.3,0,No Game
8757,2025-10-31,22:00,5002,61096.57,3752,1250,12.21,2025103122,11.6,0.0,20.6,0,No Game


## 3.4 Target Variable Construction (Demand Class Definition)

In [ ]:
# create target class based on quantiles
q1 = df_integrated['total_trips'].quantile(0.25)
q3 = df_integrated['total_trips'].quantile(0.75)

def classify(trips):
    if trips < q1: return 0 # Low
    elif trips > q3: return 2 # High
    else: return 1 # Medium

df_integrated['demand_class'] = df_integrated['total_trips'].apply(classify)


         date time_hour  total_trips  sum_duration_min  member_count  \
0  2024-11-01     00:00         4023          53396.29          2888   
1  2024-11-01     01:00         2667          35169.29          1845   
2  2024-11-01     02:00         1690          22771.54          1141   
3  2024-11-01     03:00         1073          14562.98           755   
4  2024-11-01     04:00          857          10331.25           616   

   casual_count  avg_duration_min       pk_dt  temp_c  rain_mm  wind_kmh  \
0          1135             13.27  2024110100    22.8      0.0      18.6   
1           822             13.19  2024110101    22.2      0.0      20.8   
2           549             13.47  2024110102    21.8      0.0      21.2   
3           318             13.57  2024110103    21.6      0.0      22.2   
4           241             12.06  2024110104    21.3      0.0      24.6   

   is_knicks_event         opponent  demand_class  
0                1       Miami Heat             1  
1     

In [ ]:
df_integrated

,date,time_hour,total_trips,sum_duration_min,member_count,casual_count,avg_duration_min,pk_dt,temp_c,rain_mm,wind_kmh,is_knicks_event,opponent,demand_class
0,2024-11-01,00:00,4023,53396.29,2888,1135,13.27,2024110100,22.8,0.0,18.6,1,Miami Heat,1
1,2024-11-01,01:00,2667,35169.29,1845,822,13.19,2024110101,22.2,0.0,20.8,1,Miami Heat,1
2,2024-11-01,02:00,1690,22771.54,1141,549,13.47,2024110102,21.8,0.0,21.2,1,Miami Heat,1
3,2024-11-01,03:00,1073,14562.98,755,318,13.57,2024110103,21.6,0.0,22.2,1,Detroit Pistons,0
4,2024-11-01,04:00,857,10331.25,616,241,12.06,2024110104,21.3,0.0,24.6,1,Houston Rockets,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8754,2025-10-31,19:00,8910,105210.18,7239,1671,11.81,2025103119,13.4,0.0,25.7,0,No Game,2
8755,2025-10-31,20:00,7099,85519.13,5601,1498,12.05,2025103120,13.0,0.0,27.4,0,No Game,1
8756,2025-10-31,21:00,5735,70232.43,4401,1334,12.25,2025103121,12.2,0.0,26.3,0,No Game,1
8757,2025-10-31,22:00,5002,61096.57,3752,1250,12.21,2025103122,11.6,0.0,20.6,0,No Game,1


# 4. Storage
We execute the merges and save the final result.

In [ ]:
#upsert storage
def save_incremental_to_sqlite(df_new, db_name='nyc_mobility.db'):
    conn = sqlite3.connect(db_name)
    try:
        df_old = pd.read_sql("SELECT * FROM abt_classification", conn)
    except:
        df_old = pd.DataFrame()
    
    # union and deduplication
    df_combined = pd.concat([df_old, df_new])
    df_final_storage = df_combined.drop_duplicates(subset=['pk_dt'], keep='last')
    
    df_final_storage.to_sql('abt_classification', conn, if_exists='replace', index=False)
    conn.close()
    print(f"Salvataggio DB Completato. Righe totali: {len(df_final_storage)}")

save_incremental_to_sqlite(df_integrated)

In [ ]:
# SQLlite analysis queries

# Connect to the newdatabase
conn = sqlite3.connect('nyc_mobility.db')


# QUERY 1: Validation of the signal feature (is_knicks_event)
#  showing that if there's a Knicks game, trips increase.

query_signal = """
SELECT 
    is_knicks_event, 
    COUNT(*) as ore_totali,
    ROUND(AVG(total_trips), 0) as media_viaggi_orari
FROM abt_classification
GROUP BY is_knicks_event;
"""

df_q1 = pd.read_sql(query_signal, conn)
# Mappiamo 0/1 per renderlo leggibile nella stampa
df_q1['descrizione'] = df_q1['is_knicks_event'].map({0: 'No Evento', 1: 'Evento Knicks'})
print(df_q1[['descrizione', 'ore_totali', 'media_viaggi_orari']].to_string(index=False))


# QUERY 2: chack meteorological impact on demand
# how much did the request vary based on weather conditions and demand class
query_weather = """
SELECT 
    CASE 
        WHEN rain_mm = 0 THEN 'Sereno'
        WHEN rain_mm > 0 AND rain_mm < 2 THEN 'Pioggia Leggera'
        ELSE 'Pioggia Forte'
    END as condizione_meteo,
    ROUND(AVG(total_trips), 0) as media_viaggi,
    demand_class
FROM abt_classification
GROUP BY condizione_meteo, demand_class
ORDER BY media_viaggi DESC;
"""
df_q2 = pd.read_sql(query_weather, conn)
print(df_q2.head(10).to_string(index=False))

# Chiudiamo la connessione
conn.close()
